<div style="background-color: #F9F9F9; border: 2px solid #8E44AD; border-radius: 10px; padding: 15px; margin: 20px 0;">
    <h2 style="color: #8E44AD; font-size: 30px; font-weight: bold;">Glimpse of the Notebook</h2>
    <p style="font-size: 18px; color: #7F8C8D;">
        This notebook presents a comprehensive analysis and predictive modeling pipeline to estimate <b>delivery times</b> based on historical data. Key steps include:
    </p>
    <ul style="font-size: 18px; color: #7F8C8D; margin-left: 20px;">
        <li><b>Data Cleaning and Preprocessing:</b> Addressed missing values and removed outliers using the <b>Interquartile Range (IQR)</b> method to ensure data quality.</li>
        <li><b>Dimensionality Reduction with PCA:</b> Applied <b>Principal Component Analysis (PCA)</b> to reduce feature dimensions while retaining 95% of the variance, ensuring computational efficiency and minimizing overfitting risks.</li>
        <li><b>Pipeline Creation:</b> Built a reusable pipeline that integrates preprocessing, feature scaling, PCA, and machine learning models for a seamless workflow.</li>
        <li><b>Model Training and Evaluation:</b> Trained various regression models, including <b>CatBoost, XGBoost, Gradient Boosting, Random Forest,</b> and ensemble methods such as <b>Stacking</b> and <b>Voting Regressors</b>.</li>
        <li><b>Insights and Results:</b> Evaluated models on metrics such as <b>MSE</b>, <b>MAE</b>, and <b>R²</b>, with <b>CatBoost Regressor</b> emerging as the best-performing model.</li>
    </ul>
</div>


# Step 1: Import Libraries

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
from scipy import stats
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import scatter_matrix
import sweetviz as sv
from tqdm import tqdm


# Preprocessing
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.decomposition import PCA


# Models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensemble
from sklearn.ensemble import VotingRegressor, StackingRegressor

# Others
import warnings
warnings.filterwarnings('ignore')

: 

# Load and Explore Dataset


In [ ]:
df = pd.read_csv('DoorDashData.csv')

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
df.isnull().sum()

In [ ]:
# df['year'] = df['created_at'].str[0:4]
# df['created_at'].str[5:7].unique()

# Data Cleaning

In [ ]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

In [ ]:
df['delivery_time_minutes'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds() / 60

In [ ]:
df = df.drop(columns=['market_id', 'created_at', 'actual_delivery_time', 'store_id'], axis=1)

In [ ]:
df.head(2)

In [ ]:
df.nunique()

In [ ]:
df['delivery_time_minutes'].describe().T

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

In [ ]:
df['delivery_time_minutes'].isnull().sum()

In [ ]:
df = df.iloc[:]

In [ ]:
df['delivery_time_minutes'].dropna()

In [ ]:
df.dropna(inplace=True)

<div style="background-color: #F9F9F9; border: 2px solid #3498DB; border-radius: 10px; padding: 15px; margin: 20px 0;">
    <h2 style="color: #2980B9; font-size: 30px; font-weight: bold;">3. Exploratory Data Analysis and Feature Engineering </h2>
    <p style="font-size: 18px; color: #7F8C8D;">
        Performing exploratory data analysis + Creating and transforming features
    </p>
</div>


In [ ]:
my_report = sv.analyze(df,'delivery_time_minutes')
my_report.show_notebook(w="100%", h="full") 

In [ ]:
df['dasher_latecy_rate'] = df['total_busy_dashers'] / df['total_onshift_dashers']
df['delay_time'] = df['estimated_store_to_consumer_driving_duration'] + df['estimated_order_place_duration']

In [ ]:
df.drop(inplace=True, axis=1, columns=['total_busy_dashers','total_onshift_dashers','estimated_store_to_consumer_driving_duration', 'estimated_order_place_duration'])

In [ ]:
df.shape

In [ ]:
def remove_outliers_iqr(df):
    df_cleaned = df.copy()
    
    for column in df_cleaned.select_dtypes(include=['float64', 'int64']).columns:
        Q1 = df_cleaned[column].quantile(0.25)
        Q3 = df_cleaned[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned

df_cleaned = remove_outliers_iqr(df)

In [ ]:
df_cleaned.shape

In [ ]:
# numerical columns
numerical_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns
# categorical columns
categorical_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns

In [ ]:
# # Histogram for numerical columns
# for col in numerical_cols:
#     plt.figure(figsize=(6, 4))
#     sns.histplot(df[col], bins=30, kde=True, color='skyblue')
#     plt.title(f'Distribution of {col}')
#     plt.xlabel(col)
#     plt.ylabel('Frequency')
#     plt.show()

In [ ]:
# Boxplots for outliers detection
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=df_cleaned[col], color='green')
    plt.title(f'Boxplot of {col}')
    plt.xlabel(col)
    plt.show()

In [ ]:
plt.figure(figsize=(20, 8))
df_cleaned['store_primary_category'].value_counts().head(40).plot(kind='bar')
plt.title('Top 40 Store Primary Categories', fontsize=16)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

> Lets consider top 40 categories are separate columns and and categories 

In [ ]:
def calculate_category_percentages(df):
    results = []    
    for column in df.select_dtypes(include=['object']).columns:
        category_counts = df[column].value_counts(normalize=True) * 100        
        percentage_df = category_counts.reset_index()
        percentage_df.columns = ['Category', 'Percentage']
        percentage_df['Column'] = column  
        results.append(percentage_df)
    final_result = pd.concat(results, ignore_index=True)
    return final_result

category_percentages = calculate_category_percentages(df_cleaned)

print("Percentage of Each Class in All Categorical Columns:")
category_percentages.head(40)

In [ ]:
df_cleaned.describe().T

<div style="background-color: #F9F9F9; border: 2px solid #3498DB; border-radius: 10px; padding: 15px; margin: 20px 0;">
    <h2 style="color: #2980B9; font-size: 30px; font-weight: bold;">4. Model Training</h2>
    <p style="font-size: 18px; color: #7F8C8D;">
        Splitting the dataset, training machine learning models, and evaluating their performance.
    </p>
</div>


In [ ]:
y = df_cleaned['delivery_time_minutes']
X = df_cleaned.drop(['delivery_time_minutes'],axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Advanced Feature Engineering
df_cleaned['subtotal_complexity'] = df_cleaned['subtotal'] * df_cleaned['order_complexity']
df_cleaned['items_ratio'] = df_cleaned['total_items'] / (df_cleaned['num_distinct_items'] + 1)
df_cleaned['subtotal_per_item'] = df_cleaned['subtotal'] / (df_cleaned['total_items'] + 1)
df_cleaned['max_price_per_item'] = df_cleaned['max_item_price'] / (df_cleaned['total_items'] + 1)
df_cleaned['hour_of_day'] = df_cleaned['created_at'].dt.hour
df_cleaned['day_of_week'] = df_cleaned['created_at'].dt.dayofweek
df_cleaned['log_subtotal'] = np.log1p(df_cleaned['subtotal'])
df_cleaned['log_order_complexity'] = np.log1p(df_cleaned['order_complexity'])


In [ ]:
categorical_features = ['store_primary_category']
numerical_features = [
    'order_protocol', 'total_items', 'subtotal', 'num_distinct_items',
    'min_item_price', 'max_item_price', 'total_outstanding_orders', 'dasher_latecy_rate', 'delay_time'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[ N  
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

In [ ]:
def create_pipeline(model):
    return Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),  # Retain 95% variance
        ('model', model)
    ])

In [ ]:
models = {
    'RandomForest': RandomForestRegressor(),
    # 'GradientBoosting': GradientBoostingRegressor(),
    # 'XGBoost': XGBRegressor(),
    # 'CatBoost': CatBoostRegressor(verbose=0),
    'Linear Regression': LinearRegression()    
}

In [ ]:
results = {}
for name, model in tqdm(models.items(), desc="Evaluating Models"):
    pipeline = create_pipeline(model)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MSE': mse, 'MAE': mae, 'R^2': r2}

for model_name, metrics in results.items():
    print(f"{model_name}: MSE={metrics['MSE']:.2f}, MAE={metrics['MAE']:.2f}, R^2={metrics['R^2']:.2f}")

### Summary of Results

The evaluation of regression models for predicting **delivery_time_minutes** revealed the following insights:

1. **CatBoost Regressor** emerged as the best-performing model with:
   - **MSE:** 155.54 (lowest among all models)
   - **MAE:** 9.90 (lowest among all models)
   - **R²:** 0.21 (highest among all models)

2. **Gradient Boosting Regressor** and **XGBoost Regressor** achieved comparable results, with an **R²** of 0.19, slightly behind CatBoost, and MAE values of 10.01 and 10.03, respectively.

3. **Random Forest Regressor** performed reasonably well with:
   - **MSE:** 160.87
   - **MAE:** 10.10
   - **R²:** 0.18

4. **Ridge Regression** achieved modest results with an **R²** of 0.17, while **Lasso Regression** and **AdaBoost Regressor** performed the least effectively, with an **R²** of 0.14.

In [ ]:
def create_ensemble_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

In [ ]:
best_models = {
    'CatBoost': CatBoostRegressor(verbose=0),
    'GradientBoosting': GradientBoostingRegressor(),
    # 'XGBoost': XGBRegressor(eval_metric='rmse'),
    # 'RandomForest': RandomForestRegressor()
}

In [ ]:
stacking_regressor = StackingRegressor(
    estimators=[
        ('catboost', best_models['CatBoost']),
        ('gradient_boosting', best_models['GradientBoosting']),
        # ('xgboost', best_models['XGBoost']),
        # ('random_forest', best_models['RandomForest'])
    ],
    final_estimator=Ridge()
)

voting_regressor = VotingRegressor(
    estimators=[
        ('catboost', best_models['CatBoost']),
        ('gradient_boosting', best_models['GradientBoosting']),
        # ('xgboost', best_models['XGBoost']),
        # ('random_forest', best_models['RandomForest'])
    ]
)

In [ ]:
ensemble_models = {
    'Stacking': create_ensemble_pipeline(stacking_regressor),
    'Voting': create_ensemble_pipeline(voting_regressor)
}

In [ ]:
ensemble_results = {}
for name, pipeline in tqdm(ensemble_models.items()):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    ensemble_results[name] = {'MSE': mse, 'MAE': mae, 'R^2': r2}

print("\nEnsemble Results:")
for model_name, metrics in ensemble_results.items():
    print(f"{model_name}: MSE={metrics['MSE']}, MAE={metrics['MAE']}, R^2={metrics['R^2']}")

### Summary

The ensemble model results demonstrate the performance of both the Stacking and Voting regressors in predicting delivery time in minutes. The following metrics were obtained:

- **Stacking Regressor**:
  - **Mean Squared Error (MSE)**: 150.41
  - **Mean Absolute Error (MAE)**: 9.72
  - **R-squared (R²)**: 0.23
  
  The Stacking Regressor exhibited a lower MSE and MAE compared to the Voting Regressor, indicating that it made more accurate predictions on average. The R² value of 0.23 suggests that approximately 23% of the variance in the delivery time can be explained by the model, showing a moderate level of explanatory power.

- **Voting Regressor**:
  - **Mean Squared Error (MSE)**: 151.76
  - **Mean Absolute Error (MAE)**: 9.78
  - **R-squared (R²)**: 0.23
  
  The Voting Regressor demonstrated slightly higher MSE and MAE than the Stacking Regressor, which suggests that it performed marginally less well in terms of prediction accuracy. The R² value is similar to that of the Stacking model, indicating a comparable level of explanatory power.

> Based on the performance metrics from all previous results, the CatBoost Regressor stands out as the best individual model for delivery time prediction task, achieving the highest R² value of 0.21, the lowest MSE of 155.54, and the lowest MAE of 9.90. Its performance indicates that it can effectively capture the relationships in the data.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95)) 
        ]), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

In [ ]:
catboost_model = CatBoostRegressor(verbose=0)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', catboost_model)
])

In [ ]:
pipeline.fit(X, y)

In [ ]:
def predict_delivery_time(input_data):
    """
    Predict the delivery time in minutes for given input features.

    Parameters:
    input_data (dict): A dictionary containing the feature values.

    Returns:
    float: Predicted delivery time in minutes.
    """
    input_df = pd.DataFrame([input_data])
    prediction = pipeline.predict(input_df)
    return prediction[0]

In [ ]:
example_input = {
    'store_primary_category': 'sandwich',  
    'order_protocol': 1.0,
    'total_items': 3,
    'subtotal': 30,
    'num_distinct_items': 2,
    'min_item_price': 5,
    'max_item_price': 15,
    'total_outstanding_orders': 2.0,
    'dasher_latecy_rate': 0.05,
    'delay_time': 2.0
}

predicted_time = predict_delivery_time(example_input)
print(f"Predicted Delivery Time: {predicted_time:.2f} minutes")

<div style="background-color: #F4F6F6; border: 2px solid #2ECC71; border-radius: 10px; padding: 15px; margin: 20px 0;">
    <h2 style="color: #27AE60; font-size: 30px; font-weight: bold;">Conclusion</h2>
    <p style="font-size: 18px; color: #7F8C8D;">
        The evaluation of various regression models for predicting delivery times provided several insights. The <b>CatBoost Regressor</b> emerged as the most effective model, achieving the lowest Mean Squared Error (MSE) and Mean Absolute Error (MAE), along with the highest R² value among all models. Ensemble models, such as the <b>Stacking Regressor</b>, demonstrated improved accuracy compared to individual models, showcasing the benefits of combining predictions.
    </p>
    <p style="font-size: 18px; color: #7F8C8D;">
        Models like <b>Gradient Boosting Regressor</b> and <b>XGBoost Regressor</b> also delivered competitive results, while traditional linear models such as <b>Ridge Regression</b> and <b>Lasso Regression</b> exhibited relatively lower performance. The findings highlight the importance of ensemble techniques and advanced machine learning models in predicting complex outcomes like delivery times.
    </p>
</div>
